In [1]:
import os
import sys
from types import SimpleNamespace   
from pathlib import Path
from tqdm import tqdm

import numpy as np

from scene import Scene, GaussianModel
from scene.dataset_readers import SceneInfo, CameraInfo, getNerfppNorm, fetchPly, storePly
from scene.dataset import FourDGSdataset
from scene.WAT_dataset import ColmapDataset_NGPA
from utils.graphics_utils import focal2fov, fov2focal

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### WAT dataset original

In [2]:
hparams = {
    "root_dir": "../../datasets/WAT/breville",
    "dataset_name": "colmap_ngpa_render",
    "exp_name": "breville",
    "downsample": 1.0,
    "num_epochs": 20,
    "batch_size": 8192,
    "lr": 1e-2,
    "eval_lpips": True,
    "task_curr": 4,
    "task_number": 5,
    "dim_a": 48,
    "dim_g": 16,
    "scale": 8.0,
    "vocab_size": 5,
    "weight_path": "ckpts/NGPGv2/colmap_ngpa/breville/epoch=19-v6.ckpt",
    "ray_sampling_strategy": "all_images",
    "render_fname": "UB",
    "val_only": True,
    "use_exposure": False,
}
hparams = SimpleNamespace(**hparams)

In [3]:
kwargs = {'root_dir': hparams.root_dir,
        'downsample': hparams.downsample}
train_dataset = ColmapDataset_NGPA(split='train', **kwargs)

self.img_wh = (1920, 1440)
[test] near_far = 0.2990965247154236/60.96563720703125, scale = 7.620704650878906
Preparing train split: 233 images ...


In [4]:
# From train_NGPGv2.NerfSystem.setup
train_dataset.batch_size = hparams.batch_size
train_dataset.ray_sampling_strategy = hparams.ray_sampling_strategy

In [5]:
train_dataset[0]

(tensor([[[124, 125, 128],
          [124, 125, 128],
          [124, 125, 128],
          ...,
          [244, 218, 168],
          [244, 218, 168],
          [244, 218, 168]],
 
         [[124, 125, 128],
          [124, 125, 128],
          [124, 125, 128],
          ...,
          [245, 219, 169],
          [245, 219, 169],
          [245, 219, 169]],
 
         [[124, 125, 128],
          [124, 125, 128],
          [124, 125, 128],
          ...,
          [247, 221, 171],
          [247, 221, 171],
          [247, 221, 171]],
 
         ...,
 
         [[194, 150,  79],
          [194, 150,  79],
          [187, 143,  72],
          ...,
          [116,  92,  70],
          [121,  97,  75],
          [123,  99,  77]],
 
         [[172, 134,  70],
          [171, 133,  69],
          [166, 128,  64],
          ...,
          [116,  92,  70],
          [123,  99,  77],
          [126, 102,  80]],
 
         [[155, 117,  53],
          [156, 118,  54],
          [152, 114,  50],
   

In [5]:
len(train_dataset)

233

### Integration to 4DGS

In [6]:
def format_infos(dataset):
    cameras = []

    for idx in tqdm(range(len(dataset))):
        img, pose, time = dataset[idx]
        
        # Extract image path and name from dataset
        image_path = dataset.img_paths[idx]
        image_name = os.path.basename(image_path)
        
        # Extract R and T from the pose
        pose = pose.numpy()
        R = pose[:3, :3].T  # Transpose to convert from world-to-camera to camera-to-world
        T = -R @ pose[:3, 3]  # Convert position to translation
        
        # Calculate FovX and FovY
        fx, fy = dataset.K[0, 0].item(), dataset.K[1, 1].item()
        FovX = focal2fov(fx, dataset.img_wh[0])
        FovY = focal2fov(fy, dataset.img_wh[1])
        
        # Create CameraInfo object
        camera_info = CameraInfo(
            uid=idx,
            R=R,
            T=T,
            FovY=FovY,
            FovX=FovX,
            image=img.numpy(),  # This is the actual image tensor
            image_path=image_path,
            image_name=image_name,
            width=dataset.img_wh[0],
            height=dataset.img_wh[1],
            time=time.item(),  # Convert from tensor to scalar
            mask=None  # Set to None as masks are not available in the current dataset
        )
        
        cameras.append(camera_info)

    return cameras

In [8]:
def readWATInfo(datadir):
    # Create train and test datasets
    train_dataset = ColmapDataset_NGPA(
        root_dir=datadir,
        split='train',
        downsample=1.0,
    )
    print(len(train_dataset))
    
    test_dataset = ColmapDataset_NGPA(
        root_dir=datadir,
        split='test',
        downsample=1.0,
    )
    print(len(test_dataset))

    # Format the camera information
    train_cam_infos = format_infos(train_dataset)
    test_cam_infos = format_infos(test_dataset)

    # Get NeRF++ normalization
    nerf_normalization = getNerfppNorm(train_cam_infos)

    # Check if PLY file exists, if not, create it
    ply_path = os.path.join(datadir, "sparse/0/points3D.ply")
    if not os.path.exists(ply_path):
        print("Converting point3d data to .ply, will happen only the first time you open the scene.")
        # Use the points3d data from the dataset
        xyz = train_dataset.pts3d
        # Assuming RGB values are stored in the dataset, otherwise use a default color
        if hasattr(train_dataset, 'pts3d_rgb'):
            rgb = train_dataset.pts3d_rgb
        else:
            rgb = np.zeros_like(xyz) # Default color is black
        storePly(ply_path, xyz, rgb)

    # Fetch the point cloud
    try:
        pcd = fetchPly(ply_path)
    except:
        print("Failed to fetch PLY file. Using points from dataset.")
        pcd = train_dataset.pts3d
        
    print("Number of points:", pcd.points.shape[0])

    # Calculate max_time
    all_times = np.concatenate([train_dataset.ts.numpy(), test_dataset.ts.numpy()])
    max_time = np.max(all_times)

    # Create SceneInfo object
    scene_info = SceneInfo(
        point_cloud=pcd,
        train_cameras=train_dataset,
        test_cameras=test_dataset,
        video_cameras=test_cam_infos,
        nerf_normalization=nerf_normalization,
        ply_path=ply_path,
        maxtime=max_time
    )
    return scene_info

In [9]:
scene_info = readWATInfo(hparams.root_dir)

self.img_wh = (1920, 1440)
[test] near_far = 0.2990965247154236/60.96563720703125, scale = 7.620704650878906
Preparing train split: 233 images ...
233
self.img_wh = (1920, 1440)
[test] near_far = 0.2990965247154236/60.96563720703125, scale = 7.620704650878906
Preparing test split: 34 images ...
34


100%|██████████| 34/34 [00:02<00:00, 15.74it/s]

Number of points: 122614


In [10]:
scene_info

SceneInfo(point_cloud=BasicPointCloud(points=array([[ 2.4069743e-01, -7.5761601e-03,  4.1025085e-04],
       [ 8.0547446e-01,  9.3031690e-02, -1.4609318e-02],
       [-1.0181209e-01, -1.2029129e-01, -2.3507109e-01],
       ...,
       [ 5.6650758e-02,  3.1142324e-01, -4.3375501e-01],
       [ 3.2446447e-01,  3.9816615e-01, -5.1229590e-01],
       [ 5.7407147e-01,  5.7725739e-01, -2.8934786e-01]], dtype=float32), colors=array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       ...,
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32), normals=array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       ...,
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32)), train_cameras=<scene.WAT_dataset.ColmapDataset_NGPA object at 0x7fd608145d30>, test_cameras=<scene.WAT_dataset.ColmapDataset_NGPA object at 0x7fd608425c70>, video_cameras=[CameraInfo(uid=0, R=array([[ 0.8249846 , -0.25272182,  0.5055018 ],
       [ 0.2057989

In [11]:
train_camera = FourDGSdataset(scene_info.train_cameras, None, 'WAT')

In [12]:
train_camera[0].time

array(0, dtype=int32)